# PyTorch Tutorial 43: LLM Evaluation Harness Engineering

**Author:** Anikhet Mulky  
**Date:** 2025  
**Target Role:** xAI - Coding Agents / Post-Training RL / Evals

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Implement pass@k** from scratch with the exact combinatorial formula used in production
2. **Build a code benchmark system** with problem definitions, canonical solutions, and test harnesses
3. **Create a sandboxed code execution engine** with security guardrails and timeout enforcement
4. **Design a full assessment pipeline** that orchestrates generation, execution, and metric computation
5. **Implement Elo rating tournaments** for pairwise model comparison on leaderboards
6. **Apply bootstrap methods** for confidence intervals and statistical significance testing
7. **Detect benchmark contamination** using n-gram overlap analysis
8. **Build regression dashboards** that track model quality across versions and flag degradations

---

*Every implementation is from scratch. No external ML libraries beyond PyTorch, NumPy, and Matplotlib.*

## Vocabulary First

Before diving into code, let's define every key term:

| Term | Definition |
|------|-----------|
| **pass@k** | Probability that at least one of k generated code samples passes all unit tests. The standard metric for code generation quality. |
| **HumanEval Benchmark** | OpenAI's benchmark of 164 hand-written Python programming problems with function signatures, docstrings, and unit tests. |
| **MBPP** | Mostly Basic Python Programming - Google's benchmark of ~1000 crowd-sourced Python problems, simpler than HumanEval. |
| **Elo Rating** | A zero-sum rating system (from chess) where models gain/lose points based on pairwise wins. Used by LMSYS Chatbot Arena. |
| **Bootstrap CI** | Confidence interval computed by resampling with replacement. Non-parametric - no distributional assumptions needed. |
| **Benchmark Contamination** | When benchmark problems or solutions leak into training data, inflating scores without genuine capability improvement. |
| **Regression Testing** | Tracking model quality across versions to detect when updates degrade performance on specific capabilities. |
| **n-gram Overlap** | Measuring text similarity by comparing sequences of n consecutive tokens. Used for contamination detection. |
| **Assessment Harness** | End-to-end system that generates, executes, scores, and reports model outputs against a benchmark. |
| **Sandbox Execution** | Running untrusted code in an isolated environment with restricted permissions, timeouts, and resource limits. |

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
import subprocess
import tempfile
import os
import json
import time
import hashlib
from collections import Counter
from math import comb

# Setup
plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)
print("Assessment Harness Engineering - Ready")

## Part 1: The pass@k Metric

The **pass@k** metric is THE standard for code generation assessment. Instead of asking "did the model get it right?", it asks: **"if we sample k completions, does at least one pass all tests?"**

### Mathematical Derivation

Given:
- `n` = total number of generated samples
- `c` = number of correct (passing) samples among n
- `k` = number of samples we select

$$\text{pass@k} = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}$$

**Intuition:** $\frac{\binom{n-c}{k}}{\binom{n}{k}}$ is the probability of selecting k samples and ALL of them being wrong. We subtract from 1 to get the probability that at least one is correct.

**Why not just compute accuracy?** Because temperature sampling is stochastic. A model might solve a problem 3 out of 10 times. pass@1 = 0.3, but pass@10 = 1.0. This captures the "best of k" capability that matters for real-world coding assistants.

In [ ]:
def pass_at_k(n: int, c: int, k: int) -> float:
    """Compute pass@k using the unbiased estimator from the Codex paper.
    
    Uses the combinatorial formula: pass@k = 1 - C(n-c, k) / C(n, k)
    This avoids the biased estimator of simply averaging k independent trials.
    
    Args:
        n: Total number of generated samples per problem.
        c: Number of correct (test-passing) samples.
        k: Number of samples to select.
    
    Returns:
        Probability that at least one of k selected samples is correct.
    """
    if c > n:
        raise ValueError(f"c ({c}) cannot exceed n ({n})")
    if k > n:
        raise ValueError(f"k ({k}) cannot exceed n ({n})")
    # If fewer wrong samples than k, guaranteed at least one correct
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)


def pass_at_k_batch(n_array: np.ndarray, c_array: np.ndarray, k: int) -> np.ndarray:
    """Vectorized pass@k for a batch of problems.
    
    Uses the numerically stable log-space computation to avoid
    overflow with large combinatorial values.
    
    Args:
        n_array: Array of total samples per problem.
        c_array: Array of correct samples per problem.
        k: Number of samples to select.
    
    Returns:
        Array of pass@k values, one per problem.
    """
    results = []
    for n_val, c_val in zip(n_array, c_array):
        if n_val - c_val < k:
            results.append(1.0)
            continue
        # Log-space computation: log(C(n-c,k)) - log(C(n,k))
        # log(C(a,b)) = sum(log(a-i) for i in range(b)) - sum(log(i+1) for i in range(b))
        log_ratio = 0.0
        for i in range(k):
            # Each term: log((n-c-i)/(n-i))
            log_ratio += np.log(n_val - c_val - i) - np.log(n_val - i)
        results.append(1.0 - np.exp(log_ratio))
    return np.array(results)


# --- Demonstrations ---
print("=== Single Problem Examples ===")
for n, c in [(10, 1), (10, 3), (10, 5), (10, 8)]:
    for k in [1, 5, 10]:
        if k <= n:
            score = pass_at_k(n, c, k)
            print(f"  n={n}, c={c}, k={k:2d} -> pass@{k} = {score:.4f}")
    print()

# Batch computation
print("=== Batch Computation ===")
n_arr = np.array([20, 20, 20, 20, 20])
c_arr = np.array([1, 5, 10, 15, 19])
for k in [1, 5, 10]:
    batch_results = pass_at_k_batch(n_arr, c_arr, k)
    print(f"  pass@{k}: {batch_results.round(4)}")

In [ ]:
# --- Visualization: pass@k curves for different correctness ratios ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: pass@k vs k for different c/n ratios
n_total = 50
k_values = np.arange(1, n_total + 1)
ratios = [0.05, 0.1, 0.2, 0.4, 0.6, 0.8]
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(ratios)))

for ratio, color in zip(ratios, colors):
    c_count = int(ratio * n_total)
    scores = [pass_at_k(n_total, c_count, k) for k in k_values]
    axes[0].plot(k_values, scores, color=color, linewidth=2,
                 label=f'c/n = {ratio:.0%} (c={c_count})')

axes[0].set_xlabel('k (samples selected)', fontsize=12)
axes[0].set_ylabel('pass@k', fontsize=12)
axes[0].set_title('pass@k Curves by Correctness Ratio (n=50)', fontsize=13)
axes[0].legend(fontsize=9, loc='lower right')
axes[0].set_ylim(-0.02, 1.05)
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)

# Right panel: Confidence bands via bootstrap resampling
# Simulate uncertainty by varying c around an expected value
n_total = 100
expected_ratios = [0.1, 0.3, 0.5]
k_range = np.arange(1, 51)

for idx, expected_r in enumerate(expected_ratios):
    # Bootstrap: sample c values from binomial distribution
    c_samples = np.random.binomial(n_total, expected_r, size=200)
    all_curves = []
    for c_s in c_samples:
        curve = [pass_at_k(n_total, c_s, k) for k in k_range]
        all_curves.append(curve)
    all_curves = np.array(all_curves)
    
    mean_curve = all_curves.mean(axis=0)
    lower = np.percentile(all_curves, 5, axis=0)
    upper = np.percentile(all_curves, 95, axis=0)
    
    color = colors[idx * 2]
    axes[1].plot(k_range, mean_curve, color=color, linewidth=2,
                 label=f'E[c/n] = {expected_r:.0%}')
    axes[1].fill_between(k_range, lower, upper, color=color, alpha=0.2)

axes[1].set_xlabel('k (samples selected)', fontsize=12)
axes[1].set_ylabel('pass@k', fontsize=12)
axes[1].set_title('pass@k with 90% Confidence Bands (n=100)', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].set_ylim(-0.02, 1.05)

plt.tight_layout()
plt.savefig('pass_at_k_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key insight: Even a 10% correct rate gives pass@10 ~ 0.65")

## Part 2: Building Code Benchmark Problems

A benchmark is a curated set of problems, each with:
- A **prompt** (function signature + docstring)
- A **canonical solution** (gold-standard implementation)
- **Test code** (unit tests that validate correctness)

We build a `BenchmarkBuilder` that validates problems before adding them - the canonical solution must pass all tests.

In [ ]:
@dataclass
class CodeProblem:
    """A single benchmark problem with prompt, solution, and tests."""
    task_id: str
    prompt: str           # Function signature + docstring shown to model
    entry_point: str      # Name of the function to implement
    canonical_solution: str  # Gold-standard implementation
    test_code: str        # Unit test assertions


class BenchmarkBuilder:
    """Builds and validates a code benchmark from individual problems.
    
    Validates each problem by running the canonical solution against
    its test code before accepting it into the benchmark.
    """
    
    def __init__(self, name: str):
        self.name = name
        self.problems: List[CodeProblem] = []
    
    def validate_problem(self, problem: CodeProblem) -> Tuple[bool, str]:
        """Run canonical solution against tests to verify correctness.
        
        Returns:
            Tuple of (is_valid, message).
        """
        # Combine prompt + canonical solution + test code
        full_code = problem.prompt + problem.canonical_solution + "\n" + problem.test_code
        try:
            # Execute in a subprocess for isolation
            result = subprocess.run(
                ['python3', '-c', full_code],
                capture_output=True, text=True, timeout=10
            )
            if result.returncode == 0:
                return True, "Canonical solution passes all tests"
            return False, f"Test failure: {result.stderr[:200]}"
        except subprocess.TimeoutExpired:
            return False, "Canonical solution timed out"
        except Exception as e:
            return False, f"Execution error: {str(e)}"
    
    def add_problem(self, problem: CodeProblem) -> bool:
        """Add a problem after validation. Returns True if accepted."""
        is_valid, msg = self.validate_problem(problem)
        if is_valid:
            self.problems.append(problem)
            print(f"  [PASS] {problem.task_id}: {msg}")
        else:
            print(f"  [FAIL] {problem.task_id}: {msg}")
        return is_valid
    
    def export_jsonl(self, filepath: str) -> None:
        """Export benchmark to JSONL format (industry standard)."""
        with open(filepath, 'w') as f:
            for p in self.problems:
                record = {
                    'task_id': p.task_id, 'prompt': p.prompt,
                    'entry_point': p.entry_point,
                    'canonical_solution': p.canonical_solution,
                    'test': p.test_code
                }
                f.write(json.dumps(record) + '\n')
        print(f"Exported {len(self.problems)} problems to {filepath}")


# --- Build our mini-benchmark with 5 problems ---
benchmark = BenchmarkBuilder("MiniCode-5")
print(f"Building benchmark: {benchmark.name}\n")

PROBLEMS = [
    CodeProblem(
        task_id="MC5/0", entry_point="fibonacci",
        prompt="def fibonacci(n: int) -> int:\n    \"\"\"Return the nth Fibonacci number (0-indexed).\"\"\"\n",
        canonical_solution="    if n <= 1:\n        return n\n    a, b = 0, 1\n    for _ in range(2, n + 1):\n        a, b = b, a + b\n    return b\n",
        test_code="assert fibonacci(0) == 0\nassert fibonacci(1) == 1\nassert fibonacci(10) == 55\nassert fibonacci(20) == 6765\n"
    ),
    CodeProblem(
        task_id="MC5/1", entry_point="is_palindrome",
        prompt="def is_palindrome(s: str) -> bool:\n    \"\"\"Check if string is a palindrome (ignore case and non-alphanumeric).\"\"\"\n",
        canonical_solution="    cleaned = ''.join(c.lower() for c in s if c.isalnum())\n    return cleaned == cleaned[::-1]\n",
        test_code="assert is_palindrome('racecar') == True\nassert is_palindrome('A man a plan a canal Panama') == True\nassert is_palindrome('hello') == False\nassert is_palindrome('') == True\n"
    ),
    CodeProblem(
        task_id="MC5/2", entry_point="two_sum",
        prompt="def two_sum(nums: list, target: int) -> list:\n    \"\"\"Return indices of two numbers that add up to target.\"\"\"\n",
        canonical_solution="    seen = {}\n    for i, num in enumerate(nums):\n        complement = target - num\n        if complement in seen:\n            return [seen[complement], i]\n        seen[num] = i\n    return []\n",
        test_code="assert two_sum([2, 7, 11, 15], 9) == [0, 1]\nassert two_sum([3, 2, 4], 6) == [1, 2]\nassert two_sum([1, 1], 2) == [0, 1]\n"
    ),
    CodeProblem(
        task_id="MC5/3", entry_point="binary_search",
        prompt="def binary_search(arr: list, target: int) -> int:\n    \"\"\"Return index of target in sorted array, or -1 if not found.\"\"\"\n",
        canonical_solution="    lo, hi = 0, len(arr) - 1\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            lo = mid + 1\n        else:\n            hi = mid - 1\n    return -1\n",
        test_code="assert binary_search([1, 3, 5, 7, 9], 5) == 2\nassert binary_search([1, 3, 5, 7, 9], 1) == 0\nassert binary_search([1, 3, 5, 7, 9], 4) == -1\nassert binary_search([], 1) == -1\n"
    ),
    CodeProblem(
        task_id="MC5/4", entry_point="max_subarray_sum",
        prompt="def max_subarray_sum(nums: list) -> int:\n    \"\"\"Find the contiguous subarray with the largest sum (Kadane's).\"\"\"\n",
        canonical_solution="    if not nums:\n        return 0\n    max_sum = current = nums[0]\n    for num in nums[1:]:\n        current = max(num, current + num)\n        max_sum = max(max_sum, current)\n    return max_sum\n",
        test_code="assert max_subarray_sum([-2,1,-3,4,-1,2,1,-5,4]) == 6\nassert max_subarray_sum([1]) == 1\nassert max_subarray_sum([-1,-2,-3]) == -1\nassert max_subarray_sum([5,4,-1,7,8]) == 23\n"
    ),
]

for problem in PROBLEMS:
    benchmark.add_problem(problem)

print(f"\nBenchmark has {len(benchmark.problems)} validated problems")

## Part 3: Code Execution Sandbox

Running untrusted LLM-generated code is **dangerous**. Our sandbox:
1. Blocks known dangerous patterns (network access, file deletion, dynamic imports)
2. Executes in subprocess isolation with strict timeouts
3. Captures stdout, stderr, and timing information

In [ ]:
@dataclass
class ExecutionResult:
    """Result of running code in the sandbox."""
    passed: bool
    output: str
    error: str
    runtime_ms: float


# Patterns that indicate potentially dangerous code
BLOCKED_PATTERNS = [
    'socket', 'requests.', 'urllib', 'http.client',
    'shutil.rmtree', '/etc/', '/proc/', 'os.remove',
    'os.system', 'subprocess', '__import__',
    'open("/etc', "open('/etc",
]


class ExecutionSandbox:
    """Sandboxed code execution engine for LLM-generated code.
    
    Uses subprocess isolation, pattern blocking, and timeouts
    to safely run untrusted code against test suites.
    """
    
    def __init__(self, timeout: int = 5):
        self.timeout = timeout
        self.blocked_patterns = BLOCKED_PATTERNS
    
    def _check_safety(self, code: str) -> Tuple[bool, str]:
        """Screen code for dangerous patterns before execution.
        
        Returns:
            Tuple of (is_safe, reason_if_blocked).
        """
        for pattern in self.blocked_patterns:
            if pattern in code:
                return False, f"Blocked pattern detected: '{pattern}'"
        return True, "Code passed safety check"
    
    def run_code_safely(self, code: str, test_code: str) -> ExecutionResult:
        """Execute code + tests in an isolated subprocess.
        
        Args:
            code: The function implementation to test.
            test_code: Assertion-based test code.
        
        Returns:
            ExecutionResult with pass/fail status and diagnostics.
        """
        # Step 1: Safety screening
        is_safe, reason = self._check_safety(code)
        if not is_safe:
            return ExecutionResult(
                passed=False, output="", error=reason, runtime_ms=0.0
            )
        
        # Step 2: Write to temp file and execute in subprocess
        full_code = code + "\n" + test_code
        start_time = time.perf_counter()
        
        try:
            with tempfile.NamedTemporaryFile(
                mode='w', suffix='.py', delete=False
            ) as f:
                f.write(full_code)
                temp_path = f.name
            
            result = subprocess.run(
                ['python3', temp_path],
                capture_output=True, text=True,
                timeout=self.timeout
            )
            elapsed_ms = (time.perf_counter() - start_time) * 1000
            
            return ExecutionResult(
                passed=(result.returncode == 0),
                output=result.stdout[:500],
                error=result.stderr[:500],
                runtime_ms=elapsed_ms
            )
        except subprocess.TimeoutExpired:
            elapsed_ms = (time.perf_counter() - start_time) * 1000
            return ExecutionResult(
                passed=False, output="",
                error=f"Timeout after {self.timeout}s",
                runtime_ms=elapsed_ms
            )
        finally:
            if os.path.exists(temp_path):
                os.unlink(temp_path)
    
    def run_batch(self, code_samples: List[str], test_code: str) -> List[ExecutionResult]:
        """Execute multiple code samples against the same tests."""
        results = []
        for i, code in enumerate(code_samples):
            result = self.run_code_safely(code, test_code)
            results.append(result)
        return results


# --- Demo: Safe and dangerous code ---
sandbox = ExecutionSandbox(timeout=5)

# Test with safe code
safe_code = "def fibonacci(n):\n    if n <= 1: return n\n    a, b = 0, 1\n    for _ in range(2, n+1): a, b = b, a+b\n    return b\n"
safe_test = "assert fibonacci(10) == 55\nassert fibonacci(0) == 0\n"
result = sandbox.run_code_safely(safe_code, safe_test)
print(f"Safe code:   passed={result.passed}, runtime={result.runtime_ms:.1f}ms")

# Test with dangerous code (should be blocked)
dangerous_code = "import socket\ndef evil(): socket.create_connection(('evil.com', 80))\n"
result = sandbox.run_code_safely(dangerous_code, "evil()")
print(f"Dangerous:   passed={result.passed}, error='{result.error}'")

# Test with wrong code (should fail tests, not be blocked)
wrong_code = "def fibonacci(n):\n    return n * 2\n"
result = sandbox.run_code_safely(wrong_code, safe_test)
print(f"Wrong code:  passed={result.passed}, has_error={bool(result.error)}")

# Test with infinite loop (should timeout)
loop_code = "def fibonacci(n):\n    while True: pass\n"
result = sandbox.run_code_safely(loop_code, "fibonacci(1)")
print(f"Inf. loop:   passed={result.passed}, error='{result.error}'")

## Part 4: Full Assessment Pipeline

This is the orchestrator that ties everything together:
1. Takes model outputs (code completions) and benchmark problems
2. Runs each completion through the sandbox
3. Computes pass@k metrics across all problems
4. Supports multi-model comparison

In [ ]:
@dataclass
class ModelResult:
    """Assessment results for a single model across all problems."""
    model_name: str
    # per_problem_results[task_id] = list of (passed: bool) for each sample
    per_problem_results: Dict[str, List[bool]] = field(default_factory=dict)
    metrics: Dict[str, float] = field(default_factory=dict)


class AssessmentPipeline:
    """End-to-end pipeline: execute model outputs, compute metrics, compare.
    
    Orchestrates the sandbox, pass@k computation, and reporting
    across multiple models and benchmark problems.
    """
    
    def __init__(self, problems: List[CodeProblem]):
        self.problems = {p.task_id: p for p in problems}
        self.sandbox = ExecutionSandbox(timeout=5)
    
    def assess_model(
        self, model_name: str, outputs: Dict[str, List[str]]
    ) -> ModelResult:
        """Run all model outputs through sandbox and record results.
        
        Args:
            model_name: Identifier for the model.
            outputs: Dict mapping task_id -> list of code completions.
        
        Returns:
            ModelResult with per-problem pass/fail data.
        """
        result = ModelResult(model_name=model_name)
        
        for task_id, completions in outputs.items():
            problem = self.problems[task_id]
            # Each completion is prompt + model's code body
            pass_flags = []
            for code in completions:
                full_code = problem.prompt + code
                exec_result = self.sandbox.run_code_safely(
                    full_code, problem.test_code
                )
                pass_flags.append(exec_result.passed)
            result.per_problem_results[task_id] = pass_flags
        
        return result
    
    def compute_metrics(self, result: ModelResult, k_values: List[int] = None) -> Dict[str, float]:
        """Compute pass@k metrics from assessment results.
        
        Args:
            result: ModelResult from assess_model().
            k_values: List of k values to compute. Defaults to [1, 5, 10].
        """
        if k_values is None:
            k_values = [1, 5, 10]
        
        metrics = {}
        for k in k_values:
            scores = []
            for task_id, flags in result.per_problem_results.items():
                n = len(flags)
                c = sum(flags)
                if k <= n:
                    scores.append(pass_at_k(n, c, k))
            if scores:
                metrics[f"pass@{k}"] = np.mean(scores)
        
        result.metrics = metrics
        return metrics
    
    def compare_models(self, results: List[ModelResult]) -> None:
        """Print side-by-side comparison table of model metrics."""
        # Header
        metric_keys = sorted(results[0].metrics.keys())
        header = f"{'Model':<20}" + "".join(f"{m:>12}" for m in metric_keys)
        print(header)
        print("-" * len(header))
        
        for r in results:
            row = f"{r.model_name:<20}"
            row += "".join(f"{r.metrics.get(m, 0.0):>12.4f}" for m in metric_keys)
            print(row)


# --- Mock model outputs (simulating what LLMs would generate) ---
# Model A: Strong model - correct on most problems
model_a_outputs = {}
for problem in benchmark.problems:
    samples = []
    for i in range(10):
        # 70% chance of using canonical solution, 30% wrong
        if np.random.random() < 0.7:
            samples.append(problem.canonical_solution)
        else:
            samples.append("    return None  # wrong\n")
    model_a_outputs[problem.task_id] = samples

# Model B: Weak model - correct on fewer problems
model_b_outputs = {}
for problem in benchmark.problems:
    samples = []
    for i in range(10):
        if np.random.random() < 0.3:
            samples.append(problem.canonical_solution)
        else:
            samples.append("    return None  # wrong\n")
    model_b_outputs[problem.task_id] = samples

# Run the pipeline
pipeline = AssessmentPipeline(benchmark.problems)

print("Assessing Model A (strong)...")
result_a = pipeline.assess_model("GPT-Strong", model_a_outputs)
metrics_a = pipeline.compute_metrics(result_a)

print("Assessing Model B (weak)...")
result_b = pipeline.assess_model("GPT-Weak", model_b_outputs)
metrics_b = pipeline.compute_metrics(result_b)

print("\n=== Model Comparison ===")
pipeline.compare_models([result_a, result_b])

## Part 5: Elo Rating System

When you have many models and pairwise comparisons (like LMSYS Chatbot Arena), **Elo ratings** provide a single scalar ranking. Each match updates both players' ratings based on the outcome vs. expectation.

**Key formula:** Expected score for player A: $E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}}$

After a match: $R_A' = R_A + K \cdot (S_A - E_A)$ where $S_A \in \{0, 0.5, 1\}$ (loss, draw, win).

In [ ]:
class EloRatingSystem:
    """Elo rating system for pairwise model comparison.
    
    Implements the standard Elo algorithm used in chess and adapted
    by LMSYS for LLM leaderboards. Tracks rating history for
    convergence analysis.
    """
    
    def __init__(self, k: float = 32.0, initial_rating: float = 1500.0):
        self.k = k
        self.initial_rating = initial_rating
        self.ratings: Dict[str, float] = {}
        # Track full history for convergence plotting
        self.history: Dict[str, List[float]] = {}
    
    def _ensure_player(self, name: str) -> None:
        """Initialize a player if not already registered."""
        if name not in self.ratings:
            self.ratings[name] = self.initial_rating
            self.history[name] = [self.initial_rating]
    
    def expected_score(self, rating_a: float, rating_b: float) -> float:
        """Compute expected score for player A against player B.
        
        Returns value in [0, 1] representing A's win probability.
        """
        return 1.0 / (1.0 + 10.0 ** ((rating_b - rating_a) / 400.0))
    
    def update_ratings(
        self, player_a: str, player_b: str, outcome: float
    ) -> Tuple[float, float]:
        """Update ratings after a match.
        
        Args:
            player_a: Name of player A.
            player_b: Name of player B.
            outcome: 1.0 if A wins, 0.0 if B wins, 0.5 for draw.
        
        Returns:
            Tuple of (new_rating_a, new_rating_b).
        """
        self._ensure_player(player_a)
        self._ensure_player(player_b)
        
        exp_a = self.expected_score(self.ratings[player_a], self.ratings[player_b])
        exp_b = 1.0 - exp_a
        
        # Elo update: R' = R + K * (actual - expected)
        self.ratings[player_a] += self.k * (outcome - exp_a)
        self.ratings[player_b] += self.k * ((1.0 - outcome) - exp_b)
        
        self.history[player_a].append(self.ratings[player_a])
        self.history[player_b].append(self.ratings[player_b])
        
        return self.ratings[player_a], self.ratings[player_b]
    
    def simulate_tournament(
        self, models: List[str], win_matrix: np.ndarray, n_games: int = 1000
    ) -> None:
        """Simulate a round-robin tournament using a win probability matrix.
        
        Args:
            models: List of model names.
            win_matrix: Matrix where win_matrix[i][j] = P(model i beats model j).
            n_games: Total number of games to simulate.
        """
        n_models = len(models)
        for _ in range(n_games):
            # Randomly select two different models
            i, j = np.random.choice(n_models, size=2, replace=False)
            # Determine outcome based on true win probability
            if np.random.random() < win_matrix[i][j]:
                outcome = 1.0  # model i wins
            else:
                outcome = 0.0  # model j wins
            self.update_ratings(models[i], models[j], outcome)
    
    def get_leaderboard(self) -> List[Tuple[str, float]]:
        """Return models sorted by rating (descending)."""
        return sorted(self.ratings.items(), key=lambda x: x[1], reverse=True)


# --- Simulate a tournament with 4 models ---
models = ["Frontier-v3", "Frontier-v2", "Mid-Tier", "Baseline"]
# Win probability matrix: row i beats column j with this probability
# Frontier-v3 is strongest, Baseline is weakest
win_matrix = np.array([
    [0.50, 0.65, 0.80, 0.90],  # Frontier-v3
    [0.35, 0.50, 0.70, 0.85],  # Frontier-v2
    [0.20, 0.30, 0.50, 0.75],  # Mid-Tier
    [0.10, 0.15, 0.25, 0.50],  # Baseline
])

elo = EloRatingSystem(k=16, initial_rating=1500)
elo.simulate_tournament(models, win_matrix, n_games=2000)

print("=== Final Leaderboard ===")
for rank, (model, rating) in enumerate(elo.get_leaderboard(), 1):
    print(f"  #{rank} {model:<15} Elo: {rating:.0f}")

In [ ]:
# --- Visualization: Elo convergence + final leaderboard ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_elo = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']

# Left panel: Rating convergence over games
for idx, model in enumerate(models):
    history = elo.history[model]
    axes[0].plot(history, color=colors_elo[idx], linewidth=1.5,
                 label=model, alpha=0.8)

axes[0].set_xlabel('Game Number', fontsize=12)
axes[0].set_ylabel('Elo Rating', fontsize=12)
axes[0].set_title('Elo Rating Convergence Over 2000 Games', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].axhline(y=1500, color='gray', linestyle='--', alpha=0.3, label='Initial')

# Right panel: Final leaderboard bar chart
leaderboard = elo.get_leaderboard()
names = [m for m, _ in leaderboard]
ratings = [r for _, r in leaderboard]
bar_colors = [colors_elo[models.index(m)] for m in names]

bars = axes[1].barh(names, ratings, color=bar_colors, edgecolor='white', height=0.6)
axes[1].set_xlabel('Elo Rating', fontsize=12)
axes[1].set_title('Final Elo Leaderboard', fontsize=13)
axes[1].axvline(x=1500, color='gray', linestyle='--', alpha=0.5, label='Baseline (1500)')

# Annotate bars with exact ratings
for bar, rating in zip(bars, ratings):
    axes[1].text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
                 f'{rating:.0f}', va='center', fontsize=11, fontweight='bold')

axes[1].set_xlim(min(ratings) - 50, max(ratings) + 80)
plt.tight_layout()
plt.savefig('elo_ratings.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 6: Statistical Significance Testing

When Model A scores 72% and Model B scores 70%, is A actually better? **Bootstrap methods** answer this without distributional assumptions:

1. **Bootstrap CI**: Resample scores with replacement to estimate the distribution of the mean
2. **Paired Bootstrap Test**: Check if the difference between two models is statistically significant

In [ ]:
def bootstrap_confidence_interval(
    scores: np.ndarray, n_bootstrap: int = 10000, ci: float = 0.95
) -> Tuple[float, float, float, np.ndarray]:
    """Compute bootstrap confidence interval for the mean.
    
    Resamples with replacement to build a distribution of the mean,
    then extracts percentile-based confidence bounds.
    
    Args:
        scores: Array of per-problem scores (0 or 1 for pass/fail).
        n_bootstrap: Number of bootstrap resamples.
        ci: Confidence level (e.g. 0.95 for 95% CI).
    
    Returns:
        Tuple of (mean, lower_bound, upper_bound, bootstrap_means).
    """
    n = len(scores)
    # Generate all bootstrap samples at once for efficiency
    # Each row is one resample of size n
    indices = np.random.randint(0, n, size=(n_bootstrap, n))
    bootstrap_means = scores[indices].mean(axis=1)
    
    alpha = (1.0 - ci) / 2.0
    lower = np.percentile(bootstrap_means, 100 * alpha)
    upper = np.percentile(bootstrap_means, 100 * (1.0 - alpha))
    
    return float(np.mean(scores)), float(lower), float(upper), bootstrap_means


def paired_bootstrap_test(
    scores_a: np.ndarray, scores_b: np.ndarray, n_bootstrap: int = 10000
) -> Tuple[float, np.ndarray]:
    """Paired bootstrap significance test between two models.
    
    Tests H0: mean(scores_a) = mean(scores_b) by bootstrapping
    the paired differences and computing a two-sided p-value.
    
    Args:
        scores_a: Per-problem scores for model A.
        scores_b: Per-problem scores for model B.
        n_bootstrap: Number of bootstrap resamples.
    
    Returns:
        Tuple of (p_value, bootstrap_diffs).
    """
    assert len(scores_a) == len(scores_b), "Must have paired scores"
    n = len(scores_a)
    
    # Compute paired differences
    diffs = scores_a - scores_b
    observed_diff = np.mean(diffs)
    
    # Bootstrap the differences under H0 (centered at 0)
    centered_diffs = diffs - observed_diff
    indices = np.random.randint(0, n, size=(n_bootstrap, n))
    bootstrap_diffs = centered_diffs[indices].mean(axis=1)
    
    # Two-sided p-value: fraction of bootstraps as extreme as observed
    p_value = np.mean(np.abs(bootstrap_diffs) >= np.abs(observed_diff))
    
    return float(p_value), bootstrap_diffs + observed_diff


# --- Demo with simulated assessment scores ---
np.random.seed(42)
n_problems = 164  # Same size as HumanEval benchmark

# Model A: 72% accuracy, Model B: 65% accuracy
scores_a = np.random.binomial(1, 0.72, size=n_problems).astype(float)
scores_b = np.random.binomial(1, 0.65, size=n_problems).astype(float)

# Bootstrap CIs
mean_a, lo_a, hi_a, boot_a = bootstrap_confidence_interval(scores_a)
mean_b, lo_b, hi_b, boot_b = bootstrap_confidence_interval(scores_b)

print(f"Model A: mean={mean_a:.3f}, 95% CI=[{lo_a:.3f}, {hi_a:.3f}]")
print(f"Model B: mean={mean_b:.3f}, 95% CI=[{lo_b:.3f}, {hi_b:.3f}]")

# Paired test
p_val, boot_diffs = paired_bootstrap_test(scores_a, scores_b)
print(f"\nPaired bootstrap p-value: {p_val:.4f}")
print(f"Significant at alpha=0.05? {'YES' if p_val < 0.05 else 'NO'}")

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Bootstrap distributions with CI markers
axes[0].hist(boot_a, bins=60, alpha=0.6, color='#3498db', label='Model A', density=True)
axes[0].hist(boot_b, bins=60, alpha=0.6, color='#e74c3c', label='Model B', density=True)
axes[0].axvline(lo_a, color='#3498db', linestyle='--', alpha=0.8)
axes[0].axvline(hi_a, color='#3498db', linestyle='--', alpha=0.8)
axes[0].axvline(lo_b, color='#e74c3c', linestyle='--', alpha=0.8)
axes[0].axvline(hi_b, color='#e74c3c', linestyle='--', alpha=0.8)
axes[0].set_xlabel('Mean Score', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Bootstrap Distributions with 95% CI', fontsize=13)
axes[0].legend(fontsize=11)

# Right: Paired difference distribution
axes[1].hist(boot_diffs, bins=60, alpha=0.7, color='#9b59b6', density=True)
axes[1].axvline(0, color='red', linewidth=2, linestyle='--', label='H0: diff = 0')
axes[1].axvline(np.mean(scores_a) - np.mean(scores_b), color='green',
                linewidth=2, label=f'Observed diff = {np.mean(scores_a)-np.mean(scores_b):.3f}')
axes[1].set_xlabel('Score Difference (A - B)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title(f'Paired Bootstrap Test (p = {p_val:.4f})', fontsize=13)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('bootstrap_significance.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 7: Contamination Detection

**Benchmark contamination** is one of the biggest threats to trustworthy assessment. If benchmark problems leak into training data, models memorize answers instead of demonstrating genuine reasoning.

**Approach:** Build an n-gram index of all benchmark texts, then check training documents for high overlap. Standard practice uses 13-grams (following GPT-4 technical report methodology).

In [ ]:
class ContaminationDetector:
    """Detects benchmark contamination in training data using n-gram overlap.
    
    Builds an index of n-grams from benchmark texts, then checks
    training documents for suspicious overlap. Uses word-level n-grams
    (not character-level) following the GPT-4 methodology.
    """
    
    def __init__(self, n: int = 13):
        self.n = n
        self.benchmark_ngrams: set = set()
        self.benchmark_texts: List[str] = []
    
    def _extract_ngrams(self, text: str) -> set:
        """Extract all word-level n-grams from a text.
        
        Normalizes by lowercasing and splitting on whitespace.
        Returns a set of n-gram tuples for O(1) lookup.
        """
        words = text.lower().split()
        if len(words) < self.n:
            return set()
        ngrams = set()
        for i in range(len(words) - self.n + 1):
            ngram = tuple(words[i:i + self.n])
            ngrams.add(ngram)
        return ngrams
    
    def build_ngram_index(self, benchmark_texts: List[str]) -> int:
        """Build the n-gram index from all benchmark texts.
        
        Args:
            benchmark_texts: List of benchmark problem texts.
        
        Returns:
            Total number of unique n-grams in the index.
        """
        self.benchmark_texts = benchmark_texts
        self.benchmark_ngrams = set()
        for text in benchmark_texts:
            self.benchmark_ngrams.update(self._extract_ngrams(text))
        return len(self.benchmark_ngrams)
    
    def check_contamination(self, training_text: str) -> float:
        """Check a single training document for contamination.
        
        Args:
            training_text: A document from the training corpus.
        
        Returns:
            Contamination score in [0, 1]. Higher = more contaminated.
            Score = fraction of the document's n-grams found in benchmark.
        """
        doc_ngrams = self._extract_ngrams(training_text)
        if not doc_ngrams:
            return 0.0
        # Fraction of document n-grams that appear in benchmark
        overlap = doc_ngrams.intersection(self.benchmark_ngrams)
        return len(overlap) / len(doc_ngrams)
    
    def find_contaminated_samples(
        self, training_corpus: List[str], threshold: float = 0.8
    ) -> List[Tuple[int, float]]:
        """Scan a training corpus for contaminated documents.
        
        Args:
            training_corpus: List of training documents.
            threshold: Contamination score above which a doc is flagged.
        
        Returns:
            List of (document_index, contamination_score) for flagged docs.
        """
        contaminated = []
        for idx, doc in enumerate(training_corpus):
            score = self.check_contamination(doc)
            if score >= threshold:
                contaminated.append((idx, score))
        return sorted(contaminated, key=lambda x: x[1], reverse=True)


# --- Demo: Contamination detection ---
detector = ContaminationDetector(n=8)  # Using 8-grams for shorter demo texts

# Build index from our benchmark problems
benchmark_texts = [
    p.prompt + p.canonical_solution + p.test_code
    for p in benchmark.problems
]
n_ngrams = detector.build_ngram_index(benchmark_texts)
print(f"Built n-gram index: {n_ngrams} unique 8-grams from {len(benchmark_texts)} problems\n")

# Simulate a training corpus with some contaminated and clean docs
training_corpus = [
    # Doc 0: Clean - unrelated code
    "class DataLoader:\n    def __init__(self, batch_size):\n        self.batch_size = batch_size\n"
    "    def load(self, path):\n        with open(path) as f:\n            return json.load(f)\n",
    
    # Doc 1: CONTAMINATED - copy of fibonacci problem
    benchmark_texts[0],  # Exact copy of fibonacci benchmark problem
    
    # Doc 2: Clean - different algorithm discussion
    "sorting algorithms include quicksort mergesort and heapsort each with different "
    "time complexity tradeoffs quicksort has average case n log n but worst case n squared\n",
    
    # Doc 3: CONTAMINATED - copy of palindrome problem
    benchmark_texts[1],  # Exact copy of palindrome benchmark problem
    
    # Doc 4: Clean - generic Python tutorial
    "python is a versatile programming language used in web development data science "
    "machine learning and automation it supports multiple programming paradigms\n",
]

contaminated = detector.find_contaminated_samples(training_corpus, threshold=0.3)
print("=== Contamination Scan Results ===")
for doc_idx, score in contaminated:
    print(f"  Doc {doc_idx}: contamination_score = {score:.3f} [FLAGGED]")

print("\n=== All Document Scores ===")
for idx, doc in enumerate(training_corpus):
    score = detector.check_contamination(doc)
    status = "CONTAMINATED" if score >= 0.3 else "CLEAN"
    print(f"  Doc {idx}: score = {score:.3f} [{status}]")

## Part 8: Regression Dashboard

As models evolve through versions, we need to detect **regressions** - cases where a new version is worse than the previous one on specific capabilities. This dashboard:
1. Tracks scores across multiple categories and versions
2. Flags statistically significant drops
3. Visualizes trends and alerts

In [ ]:
@dataclass
class RegressionAlert:
    """A detected regression between model versions."""
    category: str
    version: str
    previous_version: str
    score_drop: float
    current_score: float
    previous_score: float


class RegressionDashboard:
    """Tracks model quality across versions and detects regressions.
    
    Stores per-category scores for each model version and flags
    any drops exceeding a configurable threshold.
    """
    
    def __init__(self):
        # versions[version_name] = {category: score}
        self.versions: Dict[str, Dict[str, float]] = {}
        self.version_order: List[str] = []
    
    def add_version(self, version_name: str, scores: Dict[str, float]) -> None:
        """Record scores for a new model version.
        
        Args:
            version_name: Identifier like "v1.0", "v1.1", etc.
            scores: Dict mapping category name to score.
        """
        self.versions[version_name] = dict(scores)
        self.version_order.append(version_name)
    
    def detect_regressions(
        self, threshold: float = 0.05
    ) -> List[RegressionAlert]:
        """Find categories where scores dropped between consecutive versions.
        
        Args:
            threshold: Minimum absolute drop to flag as regression.
        
        Returns:
            List of RegressionAlert objects.
        """
        alerts = []
        for i in range(1, len(self.version_order)):
            prev_ver = self.version_order[i - 1]
            curr_ver = self.version_order[i]
            prev_scores = self.versions[prev_ver]
            curr_scores = self.versions[curr_ver]
            
            for category in curr_scores:
                if category in prev_scores:
                    drop = prev_scores[category] - curr_scores[category]
                    if drop >= threshold:
                        alerts.append(RegressionAlert(
                            category=category,
                            version=curr_ver,
                            previous_version=prev_ver,
                            score_drop=drop,
                            current_score=curr_scores[category],
                            previous_score=prev_scores[category]
                        ))
        return alerts
    
    def plot_trends(self) -> None:
        """Visualize score trends and regression alerts as a 2-panel figure."""
        categories = sorted(self.versions[self.version_order[0]].keys())
        alerts = self.detect_regressions(threshold=0.05)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        cat_colors = plt.cm.Set2(np.linspace(0, 1, len(categories)))
        
        # Left panel: Score trend lines per category
        for idx, cat in enumerate(categories):
            scores = [self.versions[v].get(cat, 0) for v in self.version_order]
            axes[0].plot(self.version_order, scores, 'o-', color=cat_colors[idx],
                         linewidth=2, markersize=8, label=cat)
        
        axes[0].set_xlabel('Model Version', fontsize=12)
        axes[0].set_ylabel('Score', fontsize=12)
        axes[0].set_title('Score Trends Across Versions', fontsize=13)
        axes[0].legend(fontsize=10, loc='lower left')
        axes[0].set_ylim(0, 1.05)
        axes[0].grid(True, alpha=0.3)
        
        # Right panel: Regression alerts heatmap
        # Build a matrix: rows=categories, cols=version transitions
        transitions = [
            f"{self.version_order[i]}\n->\n{self.version_order[i+1]}"
            for i in range(len(self.version_order) - 1)
        ]
        heatmap_data = np.zeros((len(categories), len(transitions)))
        
        for i in range(len(self.version_order) - 1):
            prev_v = self.version_order[i]
            curr_v = self.version_order[i + 1]
            for j, cat in enumerate(categories):
                prev_s = self.versions[prev_v].get(cat, 0)
                curr_s = self.versions[curr_v].get(cat, 0)
                # Positive = improvement, negative = regression
                heatmap_data[j, i] = curr_s - prev_s
        
        im = axes[1].imshow(heatmap_data, cmap='RdYlGn', aspect='auto',
                            vmin=-0.15, vmax=0.15)
        axes[1].set_xticks(range(len(transitions)))
        axes[1].set_xticklabels(transitions, fontsize=8)
        axes[1].set_yticks(range(len(categories)))
        axes[1].set_yticklabels(categories, fontsize=10)
        axes[1].set_title('Score Changes (Green=Better, Red=Worse)', fontsize=13)
        
        # Annotate cells with values
        for i in range(len(categories)):
            for j in range(len(transitions)):
                val = heatmap_data[i, j]
                color = 'white' if abs(val) > 0.08 else 'black'
                axes[1].text(j, i, f'{val:+.2f}', ha='center', va='center',
                             fontsize=9, fontweight='bold', color=color)
        
        plt.colorbar(im, ax=axes[1], shrink=0.8, label='Score Change')
        plt.tight_layout()
        plt.savefig('regression_dashboard.png', dpi=150, bbox_inches='tight')
        plt.show()


# --- Demo: Track 4 versions across 3 categories ---
dashboard = RegressionDashboard()

# Simulated scores: coding improves, math regresses in v1.2, reasoning dips in v1.3
dashboard.add_version("v1.0", {"Coding": 0.65, "Math": 0.70, "Reasoning": 0.72})
dashboard.add_version("v1.1", {"Coding": 0.72, "Math": 0.73, "Reasoning": 0.75})
dashboard.add_version("v1.2", {"Coding": 0.78, "Math": 0.62, "Reasoning": 0.77})
dashboard.add_version("v1.3", {"Coding": 0.82, "Math": 0.68, "Reasoning": 0.69})

# Detect and report regressions
alerts = dashboard.detect_regressions(threshold=0.05)
print("=== Regression Alerts ===")
if alerts:
    for alert in alerts:
        print(f"  REGRESSION in {alert.category}: "
              f"{alert.previous_version} ({alert.previous_score:.2f}) -> "
              f"{alert.version} ({alert.current_score:.2f}), "
              f"drop = {alert.score_drop:.2f}")
else:
    print("  No regressions detected.")

# Plot the dashboard
dashboard.plot_trends()

## FAANG Interview Questions

### Q1: How would you design a code assessment harness for a 70B parameter model?

**Answer:**

The key challenge with a 70B model is **cost and latency** - each sample takes seconds and costs real money. The architecture should optimize for:

1. **Batched Generation**: Use vLLM or TensorRT-LLM for high-throughput batch inference. Generate all n samples per problem in a single batch to maximize GPU utilization.

2. **Hierarchical Assessment**: Don't run all 164 problems at full n=200 samples. Start with a quick pass (n=10) on all problems, then allocate more samples to problems near the decision boundary (where pass@k is ambiguous).

3. **Async Sandbox Pipeline**: Decouple generation from execution. Generation is GPU-bound; execution is CPU-bound. Use a producer-consumer architecture with a message queue between them.

4. **Caching Layer**: Cache sandbox results keyed by hash(code + tests). Many LLM samples are identical or near-identical - don't re-execute duplicates.

5. **Monitoring**: Track tokens/second, cost per problem, sandbox failure rates, and timeout rates in real-time. Alert if sandbox timeout rate exceeds 5% (indicates the model is generating infinite loops).

---

### Q2: Why is pass@k preferred over simple accuracy for code generation?

**Answer:**

Simple accuracy (does the greedy/top-1 output pass?) has three critical flaws:

1. **Temperature Sensitivity**: At temperature=0, you get one deterministic output. At temperature=0.8, the same model might solve the problem 4/10 times. Accuracy conflates model capability with decoding strategy.

2. **Practical Relevance**: Real coding assistants generate multiple candidates and filter. What matters is "can the model solve it if given k attempts?" - this is exactly pass@k.

3. **Unbiased Estimation**: The pass@k formula using combinations gives an unbiased estimator. The naive approach of "run k independent trials and check if any pass" is biased because it doesn't account for the sampling-without-replacement nature of selecting k from n.

4. **Granularity**: Accuracy is binary per-problem. pass@k gives a continuous value even for a single problem, capturing "how often" the model can solve it.

---

### Q3: How do you prevent benchmark contamination in training data?

**Answer:**

This is a multi-layered defense:

1. **Pre-training Decontamination**: Before training, scan the entire corpus against benchmark n-grams (typically 13-grams). Remove or flag documents with high overlap. This is what GPT-4 and Llama papers describe.

2. **Post-hoc Detection**: After training, use canary strings - unique identifiers embedded in benchmark problems. If the model can complete a canary string, contamination is confirmed.

3. **Held-out Benchmarks**: Maintain a private, never-published benchmark set. Use it alongside public benchmarks. If a model scores much higher on public vs. private benchmarks of similar difficulty, suspect contamination.

4. **Temporal Splits**: Create benchmark problems AFTER the training data cutoff. Problems created in 2024 can't contaminate a model trained on data up to 2023.

5. **Rephrasing Tests**: Present the same problem with different variable names, docstrings, and test cases. Contaminated models show higher variance between original and rephrased versions.

---

### Q4: Elo vs. raw benchmark scores - when to use which?

**Answer:**

| Criterion | Elo Ratings | Raw Benchmark Scores |
|-----------|-------------|---------------------|
| **Best for** | Subjective tasks (chat, writing) | Objective tasks (code, math) |
| **Data source** | Human preferences / pairwise comparisons | Automated test suites |
| **Transitivity** | Not guaranteed (A > B > C doesn't mean A > C) | Guaranteed if benchmark is consistent |
| **Sample efficiency** | Needs many comparisons to converge | One run per model |
| **Gaming risk** | Hard to game (human judges vary) | Easy to contaminate |
| **Interpretability** | Relative ranking only | Absolute capability measure |

**Use Elo when:** You have subjective quality judgments, open-ended generation, or need to compare models on tasks without clear ground truth (e.g., "which response is more helpful?").

**Use raw scores when:** You have deterministic test suites, need reproducibility, or want to measure absolute capability (e.g., "what fraction of competition math problems can this model solve?").

---

### Q5: How would you assess expensive models cost-effectively?

**Answer:**

1. **Adaptive Sampling**: Start with k=1 sample per problem. If it passes, you know pass@1=1 for that problem - no need for more samples. Focus additional samples on problems that failed initially.

2. **Difficulty Stratification**: Profile your benchmark by difficulty (using historical model performance). Easy problems (90%+ pass rate across models) add little signal - sample them less. Hard problems (10-50% pass rate) are most discriminative - sample them more.

3. **Item Response Theory (IRT)**: Fit a 2-parameter IRT model from cheap model runs. Use it to predict expensive model performance with fewer samples, then confirm predictions selectively.

4. **Proxy Models**: Run a smaller, cheaper model from the same family first. Use its per-problem scores to prioritize which problems to run on the expensive model.

5. **Early Stopping**: If after 50/164 problems, the model's score is already 3 standard deviations below the previous best, abort the run. The final score is extremely unlikely to beat the baseline.

## Key Takeaways

- [ ] **pass@k is THE metric** for code generation: $\text{pass@k} = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}$. Use log-space computation for numerical stability at scale.

- [ ] **Benchmark problems need validation**: Always run canonical solutions against tests before adding to a benchmark. Invalid problems poison all downstream metrics.

- [ ] **Sandboxing is non-negotiable**: LLM-generated code can contain network calls, file deletions, or infinite loops. Use subprocess isolation, pattern blocking, and strict timeouts.

- [ ] **The assessment pipeline is a data pipeline**: Treat it with the same rigor - idempotent execution, caching, progress tracking, and error handling at every stage.

- [ ] **Elo ratings work for subjective comparisons**: When you can't define "correct" (chat quality, writing style), pairwise Elo provides a principled ranking. But it requires many comparisons to converge.

- [ ] **Always compute confidence intervals**: A 2% improvement with overlapping CIs is noise, not signal. Bootstrap methods give you CIs without distributional assumptions.

- [ ] **Contamination is the silent killer of benchmarks**: Use n-gram overlap detection at training time and hold-out benchmarks for validation. A contaminated benchmark is worse than no benchmark.

- [ ] **Track regressions systematically**: Models often improve on one capability while regressing on another. Automated regression detection catches problems that aggregate metrics miss.

---

*Next notebook: Tutorial 44 will cover RLHF and reward model training from scratch.*